# **01 - Data Preparation**

This notebook covers the end-to-end preparation for the TFG:
1. **Data Loading**: Integration of XLSX metadata and CSV text files.
2. **Multilabel Reconstruction**: Mapping SDGs to each announcement.
3. **Filtering**: Keeping only announcements with at least one SDG.
4. **Cleaning & Normalization**: Base cleaning applied to `full_text` (for EDA and DL).
5. **Export**: Saving `full_dataset.parquet` (pre-filter, for EDA) and `processed_dataset.parquet` (filtered + cleaned, for notebook 03).

## **1. Imports & Configuration**

In [1]:
import sys
import pandas as pd
import os
import re
import unicodedata
import time

sys.path.append('../src')
from utils import (
    RAW_METADATA_PATH, RAW_TEXT_CSV_DIR, OUTPUT_DIR,
    PROCESSED_ML_DIR, PROCESSED_DL_DIR, RANDOM_SEED
)
from schema import Metadata, Description, ProcessedData

os.makedirs(OUTPUT_DIR, exist_ok=True)
os.makedirs(PROCESSED_ML_DIR, exist_ok=True)
os.makedirs(PROCESSED_DL_DIR, exist_ok=True)

## **2. Data Integration**

Load XLSX metadata and merge with raw text CSVs. The `full_text` column concatenates
the announcement title and body. The unfiltered dataset is exported here for use by the EDA notebook (02).

In [2]:
# 2.1 Load Metadata
df_meta = pd.read_excel(RAW_METADATA_PATH, dtype=str)
df_meta[Metadata.ODS] = df_meta[Metadata.ODS].str[:6].str.strip()

# Aggregate multiple ODS rows per announcement into a list
df_grouped_ods = df_meta.groupby(Metadata.ID)[Metadata.ODS].apply(
    lambda x: list(set(v for v in x if pd.notna(v)))
).reset_index()
df_grouped_ods.rename(columns={Metadata.ODS: ProcessedData.ODS_LIST}, inplace=True)

df_meta_unique = df_meta.drop(columns=[Metadata.ODS]).drop_duplicates(subset=[Metadata.ID])
df_meta = df_meta_unique.merge(df_grouped_ods, on=Metadata.ID, how='left')

# 2.2 Load Texts from multiple CSVs
csv_files = [os.path.join(RAW_TEXT_CSV_DIR, f) for f in os.listdir(RAW_TEXT_CSV_DIR) if f.endswith('.csv')]
df_texts = pd.concat([pd.read_csv(f, dtype=str) for f in csv_files], ignore_index=True)

# 2.3 Merge Metadata and Texts on ID
df = df_meta.merge(
    df_texts[[Description.ID_REGISTRE, Description.TEXT]].rename(
        columns={Description.ID_REGISTRE: Metadata.ID_REGISTRE}
    ),
    on=Metadata.ID_REGISTRE, how='left'
)

# 2.4 Create full_text field (Title + Body)
df[ProcessedData.FULL_TEXT] = (
    df[Metadata.TITLE].fillna('') + " " + df[Description.TEXT].fillna('')
)

# Fill missing ODS lists with empty list
df[ProcessedData.ODS_LIST] = df[ProcessedData.ODS_LIST].apply(
    lambda x: x if isinstance(x, list) else []
)

# Export unfiltered dataset for EDA (notebook 02)
df.to_parquet(os.path.join(OUTPUT_DIR, 'full_dataset.parquet'), index=False)

print(f"Initial dataset size: {len(df):,}")
print(f"Exported: full_dataset.parquet  (pre-filter, for EDA)")

Initial dataset size: 35,520
Exported: full_dataset.parquet  (pre-filter, for EDA)


## **3. Filtering**

Keep only announcements that have at least one ODS label assigned.
Unlabelled announcements are not useful for supervised training.

In [3]:
df_filtered = df[df[ProcessedData.ODS_LIST].apply(len) > 0].copy()
print(f"Filtered dataset size (announcements with ≥1 ODS): {len(df_filtered):,}")
print(f"Removed (no label):                                 {len(df) - len(df_filtered):,}")

Filtered dataset size (announcements with ≥1 ODS): 19,284
Removed (no label):                                 16,236


## **4. Base Cleaning & Normalisation**

`clean_base` is applied to **all** paths (DL and ML). It:
- Truncates excessively long documents (≥200 k chars — affects only ~0.5 % of the corpus).
- Applies Unicode NFC normalisation to homogenise accented characters.
- Removes ASCII control characters (tabs, null bytes, etc.).
- Strips URLs.

The result is stored in `text_dl`, which is the text used by Transformer-based models.
The ML pipeline will apply further normalisation (lemmatisation, stop-word removal) in notebook 03.

In [4]:
MAX_LEN = 200_000  # character-level truncation (affects ~0.5 % of corpus)

def clean_base(text: str) -> str:
    """Minimal cleaning shared by all modelling pipelines."""
    if not isinstance(text, str) or not text.strip():
        return ''
    text = text[:MAX_LEN]
    text = unicodedata.normalize('NFC', text)          # homogenise accented chars
    text = re.sub(r'[\x00-\x1f\x7f]', ' ', text)      # remove control characters
    text = re.sub(r'https?://\S+|www\.\S+', '', text)  # strip URLs
    text = re.sub(r' {2,}', ' ', text).strip()         # collapse multiple spaces
    return text

In [5]:
start = time.time()
df_filtered[ProcessedData.FULL_TEXT] = df_filtered[ProcessedData.FULL_TEXT].apply(clean_base)
print(f"Base cleaning completed in {time.time() - start:.2f} s")

Base cleaning completed in 6.31 s


### 4.1 Derived text columns

| Column | Content | Used by |
|--------|---------|--------|
| `full_text` | After `clean_base` | EDA, intermediate |
| `text_dl`   | Same as `full_text` (kept separate for clarity) | DL Transformers |

The `text_ml` column (lemmatised, stop-word-free) is generated in notebook 03 to keep
the heavy spaCy processing close to model training.

In [6]:
start = time.time()
df_filtered[ProcessedData.TEXT_DL] = df_filtered[ProcessedData.FULL_TEXT].str.strip()
print(f"DL text prepared in {time.time() - start:.2f} s")

DL text prepared in 0.05 s


## **5. Basic Normalisation Diagnostics**

Quick sanity checks on the cleaned text to detect empty documents or unusually short texts.

In [7]:
n_empty = (df_filtered[ProcessedData.TEXT_DL].str.strip() == '').sum()
n_short = (df_filtered[ProcessedData.TEXT_DL].str.split().str.len() < 10).sum()

print(f"Empty texts after cleaning : {n_empty}")
print(f"Texts with fewer than 10 words: {n_short} ({n_short/len(df_filtered)*100:.2f}%)")
print()
print("Character length statistics (text_dl):")
print(df_filtered[ProcessedData.TEXT_DL].str.len().describe().round(1))

Empty texts after cleaning : 0
Texts with fewer than 10 words: 5 (0.03%)

Character length statistics (text_dl):
count     19284.0
mean      17277.2
std       27270.8
min          47.0
25%        1519.0
50%        3511.5
75%       26130.5
max      200000.0
Name: text_dl, dtype: float64


## **6. Export Processed Dataset**

Save the filtered, cleaned dataset as a single Parquet file. Notebook 03 reads this file,
applies the ML-specific text normalisation (spaCy lemmatisation) and performs the
stratified split + model training.

In [8]:
out_path = os.path.join(OUTPUT_DIR, 'processed_dataset.parquet')
df_filtered.to_parquet(out_path, index=False)

print(f"Exported: {out_path}")
print(f"Shape   : {df_filtered.shape}")
print(f"Columns : {df_filtered.columns.tolist()}")
print()
print("✓ Data integrated, cleaned, and exported. Ready for EDA (02) and modelling (03).")

Exported: /Users/jia/Documents/TFG/data/processed/processed_dataset.parquet
Shape   : (19284, 11)
Columns : ['ANH_ID', 'ANU_DATA_PUBLICACIO', 'ANU_NUM_REGISTRE', 'ORG_NOM', 'ANH_TITOL', 'Tipus Anunci', 'PDF', 'ods_list', 'text', 'full_text', 'text_dl']

✓ Data integrated, cleaned, and exported. Ready for EDA (02) and modelling (03).
